# Independent saved causal EEW v2 audit
Read-only inspection; outputs only in this audit directory. No original notebook executed, no full dataset generation, no model loading/training. Exhaustive saved-array/metadata checks plus a small set of source-matched deterministic counterfactuals. Generator definitions are extracted from notebook 02, not from the different pilot implementation. All classification claims are evidence-specific.


In [1]:
from pathlib import Path
import sys,ast,json,hashlib,copy
sys.dont_write_bytecode=True
from typing import Dict,Any,List,Tuple
import numpy as np
import pandas as pd
import h5py
from scipy.signal import butter,sosfilt
ROOT=(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()) / 'cnn/cnn wave')
V2=ROOT/"eew_causal_v2"; OUT=V2/"codex_independent_audit";OUT.mkdir(exist_ok=True)
DATA=V2/"dataset"; SIM=ROOT/"data wave/synthetic_waveform_simulator/output"
def readj(p):return json.loads(p.read_text(encoding="utf-8"))
def sha(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for block in iter(lambda:f.read(4*1024*1024),b""):h.update(block)
    return h.hexdigest()
def normal(v):
    if isinstance(v,dict):return {str(k):normal(x) for k,x in v.items()}
    if isinstance(v,(list,tuple,np.ndarray)):return [normal(x) for x in v]
    if isinstance(v,np.generic):return normal(v.item())
    return v
baseline=readj(V2/"original_files_manifest_initial.json")
paths={ROOT/rel for rel in baseline}
paths.update(p for p in V2.rglob("*") if p.is_file() and OUT not in p.parents)
old_audit_path=ROOT/"phase1_waveform_cnn_results/reports/waveform_audit.json"
paths.add(old_audit_path)
meta_path=SIM/"phase2_travel_time/metadata_with_theoretical_arrivals.csv"
split_path=SIM/"phase11_preprocessed_dataset/split_indices.csv"
noise_path=SIM/"phase10_classification_dataset/earthquake_noise_mapping.csv"
paths.update([meta_path,split_path,noise_path])
before={p.relative_to(ROOT).as_posix():sha(p) for p in sorted(paths)}
historical={rel:before[rel]==digest.lower() for rel,digest in baseline.items()}
old_audit=readj(old_audit_path)
params=readj(V2/"configs/causal_simulation_parameters.json")
meta=pd.read_csv(meta_path).set_index("source_id")
assert meta.index.is_unique
split_map=pd.read_csv(split_path)
noise_map=pd.read_csv(noise_path).set_index("combined_index")
assert noise_map.index.is_unique
manifest=pd.read_csv(V2/"configs/causal_eew_dataset_manifest.csv")
nb2=readj(V2/"02_generate_causal_eew_dataset.ipynb")
gen_source="".join(nb2["cells"][2]["source"])
tree=ast.parse(gen_source)
class_source=ast.unparse(ast.Module(body=[n for n in tree.body if isinstance(n,ast.ClassDef)],type_ignores=[]))
exec(compile(class_source,"saved_notebook02_generator_definitions","exec"))
# No notebook setup/generation/write cells are executed.
results={"protected_before":before,"original17_matches_initial":historical,"records":{},
         "notes":["No regression training consumer or explicit regression mask supplied; y==1 must gate all magnitude loss.",
                  "NPZ lacks IDs, arrivals and 1/2s masks; exhaustive row equality to HDF5 is checked, but safe use requires manifest/HDF5 joins."]}
print("Protected files snapshotted:",len(before),"; original manifest matches:",sum(historical.values()),"/",len(historical))


Protected files snapshotted: 41 ; original manifest matches: 17 / 17


In [2]:
all_metadata=[];hash_owner={};duplicates=[];npz_equal={};pilots=[]
mapping={"w1_norm":"windows_1s_norm","w2_norm":"windows_2s_norm","w3_norm":"windows_3s_norm",
         "w1_raw":"windows_1s_raw","w2_raw":"windows_2s_raw","w3_raw":"windows_3s_raw",
         "labels":"labels","magnitude":"magnitude","hypo_dist":"hypocentral_distance_km","pre_s_valid_3s":"pre_s_valid_3s"}
with np.load(DATA/"eew_windows_multitask.npz",allow_pickle=False) as archive:
    results["npz_keys"]=archive.files
    for split,expected in [("train",11324),("validation",2426),("test",2428)]:
        with h5py.File(DATA/f"eew_dataset_{split}.h5","r") as f:
            rec={k:f[k][:] for k in ["labels","magnitude","depth_km","hypocentral_distance_km",
                 "p_arrival_sample","s_arrival_sample","pair_id","pre_s_valid_1s","pre_s_valid_2s","pre_s_valid_3s"]}
            for key in ["source_id","trace_name","receiver_code"]:rec[key]=f[key].asstr()[:]
            df=pd.DataFrame(rec);df["split"]=split
            df["row"]=np.arange(len(df));eq=df.labels.to_numpy()==1
            old=split_map[split_map.split==split].reset_index(drop=True)
            man=manifest[manifest.split==split].reset_index(drop=True)
            df["combined_index"]=old.combined_index
            m=meta.loc[df.loc[eq,"source_id"]]
            nseed=noise_map.loc[df.loc[~eq,"combined_index"]]
            checks={"n":len(df)==expected,"fs":float(f.attrs["sampling_rate_hz"])==100.,
                "balanced_binary":set(np.unique(rec["labels"]))=={0,1} and int(eq.sum())*2==expected,
                "magnitudes_match":np.array_equal(rec["magnitude"][eq],m.Magnitude_Value.to_numpy(dtype=np.float32)),
                "noise_sentinels":bool(np.all(rec["magnitude"][~eq]==-1)&np.all(rec["p_arrival_sample"][~eq]==-1)&np.all(rec["s_arrival_sample"][~eq]==-1)),
                "source_order_matches":df.loc[eq,"source_id"].tolist()==old.loc[eq,"source_id"].tolist(),
                "pair_order_matches":np.array_equal(df.pair_id,old.pair_id),
                "labels_order_matches":np.array_equal(df.labels,old.label),
                "manifest_alignment":np.array_equal(df.pair_id,man.pair_id) and np.array_equal(df.labels,man.label) and df.source_id.tolist()==man.source_id.tolist(),
                "manifest_magnitudes":np.allclose(df.magnitude,man.magnitude,rtol=0,atol=5e-7),
                "depths_match":np.array_equal(rec["depth_km"][eq],m.Depth_km.to_numpy(dtype=np.float32)),
                "distances_match":np.array_equal(rec["hypocentral_distance_km"][eq],m.hypocentral_distance_km.to_numpy(dtype=np.float32))}
            p=rec["p_arrival_sample"][eq];s=rec["s_arrival_sample"][eq]
            dt=m.theoretical_sp_time_seconds.to_numpy()
            checks["S_quantization"]=np.array_equal(s,np.rint((p/100+dt)*100).astype(np.int32))
            checks["travel_time_units"]=bool(np.allclose(dt,m.hypocentral_distance_km.to_numpy()*(1/params["vs_km_s"]-1/params["vp_km_s"]),rtol=1e-6,atol=1e-6))
            checks["P_before_S"]=bool(np.all(p<s))
            validity={}
            for tau in [1,2,3]:
                actual=rec[f"pre_s_valid_{tau}s"]
                checks[f"mask_{tau}s_dtype"]=actual.dtype==np.dtype(bool)
                checks[f"mask_{tau}s"]=bool(np.array_equal(actual[eq],dt>tau))
                validity[str(tau)]={"eligible_eq":int(actual[eq].sum()),"eq_n":int(eq.sum()),
                    "continuous_SP_min":float(dt.min()),"index_vs_continuous_disagreements":int(np.sum((s-p>tau*100)!=(dt>tau))),
                    "noise_true_masks":int(actual[~eq].sum())}
            shapes={k:{"shape":list(f[k].shape),"dtype":str(f[k].dtype)} for k in f}
            finite=True;consistent=True;normalization_error=0.;empty=0;zero_rows=0
            # Every stored numeric value checked; no full parent waveform exists in this archive.
            for key in f:
                if f[key].dtype.kind in "biuf":
                    for start in range(0,expected,512):
                        chunk=f[key][start:start+512]
                        finite &= bool(np.isfinite(chunk).all());empty += int(chunk.size==0)
            for start in range(0,expected,256):
                end=min(expected,start+256);raw=f["windows_3s_raw"][start:end];norm=f["windows_3s_norm"][start:end]
                mu=raw[:,:100].mean(axis=1,keepdims=True)
                sd=raw[:,:100].std(axis=1,keepdims=True)+1e-8
                calculated=(raw-mu)/sd
                normalization_error=max(normalization_error,float(np.max(np.abs(calculated-norm))))
                zero_rows+=int(np.sum(np.all(raw==0,axis=1)))
                for tau,length in [(1,200),(2,300),(3,400)]:
                    for typ,array in [("raw",raw),("norm",norm)]:
                        block=f[f"windows_{tau}s_{typ}"][start:end]
                        consistent &= np.array_equal(block,array[:,:length])
                        checks[f"shape_{tau}s_{typ}"]=f[f"windows_{tau}s_{typ}"].shape==(expected,length) and f[f"windows_{tau}s_{typ}"].dtype==np.dtype("float32")
                        for local,row in enumerate(block):
                            digest=hashlib.sha256(row.tobytes()).hexdigest()
                            key=(tau,typ,digest);prior=hash_owner.get(key)
                            if prior and prior[0]!=split:duplicates.append({"first":prior,"second":[split,start+local],"kind":f"{tau}s_{typ}"})
                            else:hash_owner[key]=(split,start+local)
                    consistent &= np.array_equal(f[f"windows_pure_post_p_{tau}s"][start:end],raw[:,100:100+tau*100])
                    checks[f"shape_post_{tau}"]=f[f"windows_pure_post_p_{tau}s"].shape==(expected,tau*100)
            checks.update(finite=finite,no_empty_arrays=empty==0,no_zero_windows=zero_rows==0,
                          nested_horizons_and_post_consistent=consistent,
                          pre_context_normalization_matches=normalization_error<5e-6)
            for suffix,hkey in mapping.items():
                npz_equal[f"{split}_{suffix}"]=bool(np.array_equal(archive[f"{split}_{suffix}"],f[hkey][:]))
            df["background_key"]=["eq:"+str(42+int(i)) if label==1 else "noise:"+str(int(noise_map.loc[i,"seed"]))+":"+str(noise_map.loc[i,"noise_profile_type"])
                                  for i,label in zip(df.combined_index,df.labels)]
            all_metadata.append(df)
            # Stratified actual saved examples: weak, strong, shallow, deep, shortest S-P and both extremes of time.
            eq_rows=df[eq];chosen=set([eq_rows.magnitude.idxmin(),eq_rows.magnitude.idxmax(),
                                     eq_rows.depth_km.idxmin(),eq_rows.depth_km.idxmax(),int(eq_rows.index[np.argmin(dt)])])
            for index in sorted(chosen):
                row=df.iloc[index].to_dict()
                row["saved"]={key:f[key][index] for key in ["windows_1s_raw","windows_2s_raw","windows_3s_raw","windows_1s_norm","windows_2s_norm","windows_3s_norm"]}
                pilots.append(row)
            for index in df[~eq].index[:2]:
                row=df.iloc[index].to_dict();row["saved"]={k:f[k][index] for k in ["windows_3s_raw","windows_3s_norm"]};pilots.append(row)
            results["records"][split]={"n":expected,"eq":int(eq.sum()),"checks":checks,"shapes":shapes,
                "normalization_max_error":normalization_error,"eligibility":validity,
                "P_min_max_samples":[int(p.min()),int(p.max())],"P_std_samples":float(p.std())}
records=pd.concat(all_metadata,ignore_index=True)
split_checks={}
for first,second in [("train","validation"),("train","test"),("validation","test")]:
    left=records[records.split==first];right=records[records.split==second]
    for key in ["source_id","pair_id","trace_name","background_key"]:
        split_checks[f"{first}/{second}:{key}"]=len(set(left[key])&set(right[key]))
pair_counts=records.groupby("pair_id").agg(n=("labels","size"),earthquakes=("labels","sum"),splits=("split","nunique"))
results["splits"]={"intersections":split_checks,"all_pairs_one_eq_one_noise_same_split":bool(((pair_counts.n==2)&(pair_counts.earthquakes==1)&(pair_counts.splits==1)).all()),
 "cross_split_window_duplicates":duplicates,"window_hashes_compared":len(hash_owner),
 "limitation":"Full 100s parents not saved; background identity relies on unique documented generation keys, checked with small exact replays."}
results["npz_exact_hdf5_equality"]=npz_equal
print("Exhaustive records:",len(records),"; NPZ matches",sum(npz_equal.values()),"/",len(npz_equal),
      "; failing structural checks",[(s,k) for s,r in results["records"].items() for k,v in r["checks"].items() if not v])


Exhaustive records: 16178 ; NPZ matches 30 / 30 ; failing structural checks []


In [3]:
# Instrument only the exact saved generator class in memory; no source files are changed.
def instrument(mode):
    module=ast.parse(class_source)
    cls=next(n for n in module.body if isinstance(n,ast.ClassDef) and n.name=="CausalEEWGenerator")
    method=next(n for n in cls.body if isinstance(n,ast.FunctionDef) and n.name=="synthesize_earthquake_window")
    body=[]
    for node in method.body:
        body.append(node)
        names=[t.id for t in node.targets if isinstance(t,ast.Name)] if isinstance(node,ast.Assign) else []
        if mode=="future_noise" and "raw_noise" in names:
            body.extend(ast.parse("raw_noise[self.time_axis >= tp_sec + 8.0] *= 4.0").body)
        if mode=="future_P_coda" and "env_p" in names and isinstance(node.value,ast.BinOp) and isinstance(node.value.op,ast.Mult):
            body.extend(ast.parse("env_p[tau_p >= 4.0] *= 4.0").body)
        if mode=="future_S_coda" and "env_s" in names and isinstance(node.value,ast.BinOp) and isinstance(node.value.op,ast.Mult):
            body.extend(ast.parse("env_s[tau_s >= 5.0] *= 4.0").body)
        if isinstance(node,ast.Return):
            for key,expr in {"_full":"full_waveform","_noise":"noise","_pw":"pw","_sw":"sw",
                             "_raw_noise":"raw_noise","_filtered_noise":"filtered_noise",
                             "_mean":"mean_pre","_std":"std_pre"}.items():
                node.value.keys.append(ast.Constant(key));node.value.values.append(ast.parse(expr,mode="eval").body)
    method.body=body;ast.fix_missing_locations(module)
    scope=dict(globals());exec(compile(module,"audit_in_memory_"+mode,"exec"),scope)
    return scope["CausalEEWGenerator"]
classes={mode:instrument(mode) for mode in ["original","future_noise","future_P_coda","future_S_coda"]}
counterfactuals=[];replays=[];stratified=[]
def diff(a,b):return float(np.max(np.abs(np.asarray(a,dtype=float)-np.asarray(b,dtype=float))))
def normalize_full(r):return (r["_full"]-r["_mean"])/r["_std"]
for row in pilots:
    if row["labels"]==0:
        nr=noise_map.loc[row["combined_index"]]
        replay=CausalNoiseWindowGenerator(params).generate_noise_window(str(nr.noise_profile_type),int(nr.seed),.05)
        replays.append(dict(split=row["split"],row=row["row"],source_id=row["source_id"],type="noise",
             raw_error=diff(replay["w3_raw"],row["saved"]["windows_3s_raw"]),
             norm_error=diff(replay["w3_norm"],row["saved"]["windows_3s_norm"])))
        continue
    mr=meta.loc[row["source_id"]]
    kwargs=dict(magnitude=float(mr.Magnitude_Value),hypocentral_distance_km=float(mr.hypocentral_distance_km),
                theoretical_sp_sec=float(mr.theoretical_sp_time_seconds),seed=42+int(row["combined_index"]),target_noise_rms=.05)
    base=classes["original"](params).synthesize_earthquake_window(**kwargs)
    raw_error=max(diff(base[f"w{t}_raw"],row["saved"][f"windows_{t}s_raw"]) for t in [1,2,3])
    norm_error=max(diff(base[f"w{t}_norm"],row["saved"][f"windows_{t}s_norm"]) for t in [1,2,3])
    replays.append(dict(split=row["split"],row=row["row"],source_id=row["source_id"],type="earthquake",
                        raw_error=raw_error,norm_error=norm_error))
    s4=classes["original"](params);s4.sp_ratio*=4
    altered_s=s4.synthesize_earthquake_window(**kwargs)
    pre_s=int(np.ceil(base["ts_sec"]*100))
    cf={"source_id":row["source_id"],"seed":kwargs["seed"],"magnitude":kwargs["magnitude"],
        "depth_km":float(mr.Depth_km),"SP_seconds":kwargs["theoretical_sp_sec"],
        "Sx4_preS_raw":diff(base["_full"][:pre_s],altered_s["_full"][:pre_s]),
        "Sx4_preS_normalized":diff(normalize_full(base)[:pre_s],normalize_full(altered_s)[:pre_s]),
        "Sx4_noise_fixed":np.array_equal(base["_noise"],altered_s["_noise"]),
        "Sx4_P_fixed":np.array_equal(base["_pw"],altered_s["_pw"])}
    for mode in ["future_noise","future_P_coda","future_S_coda"]:
        changed=classes[mode](params).synthesize_earthquake_window(**kwargs)
        cutoff={"future_noise":base["tp_sec"]+8,"future_P_coda":base["tp_sec"]+4,"future_S_coda":base["ts_sec"]+5}[mode]
        stop=int(np.ceil(cutoff*100))
        cf[mode]={"cutoff_sec":cutoff,"earlier_raw_difference":diff(base["_full"][:stop],changed["_full"][:stop]),
            "earlier_normalized_difference":diff(normalize_full(base)[:stop],normalize_full(changed)[:stop]),
            "saved3s_raw_difference":diff(base["w3_raw"],changed["w3_raw"]),
            "saved3s_normalized_difference":diff(base["w3_norm"],changed["w3_norm"])}
        if mode=="future_noise":
            cf[mode]["innovations_before_cutoff_identical"]=np.array_equal(base["_raw_noise"][:stop],changed["_raw_noise"][:stop])
            cf[mode]["causal_filter_prefix_identical"]=np.array_equal(base["_filtered_noise"][:stop],changed["_filtered_noise"][:stop])
        else:cf[mode]["ambient_noise_identical"]=np.array_equal(base["_noise"],changed["_noise"])
    changed_full=base["_full"].copy();stop=base["p_sample"]+300
    changed_full[stop:]+=123
    normalized_changed=(changed_full-base["_mean"])/base["_std"]
    cf["post_generation_future_change_prefix_difference"]=diff(normalize_full(base)[:stop],normalized_changed[:stop])
    stronger=classes["original"](params).synthesize_earthquake_window(**{**kwargs,"magnitude":kwargs["magnitude"]+.5})
    cf["fixed_geometry_magnitude_plus_half_normalized3s_difference"]=diff(base["w3_norm"],stronger["w3_norm"])
    stratified.append({"source_id":row["source_id"],"magnitude":kwargs["magnitude"],"depth_km":float(mr.Depth_km),
        "raw_pre_rms":float(np.sqrt(np.mean(base["w3_raw"][:100]**2))),
        "raw_post_rms":float(np.sqrt(np.mean(base["w3_raw"][100:]**2))),
        "P_before_onset_zero":bool(np.all(base["_pw"][:base["p_sample"]]==0)),
        "S_before_onset_zero":bool(np.all(base["_sw"][:pre_s]==0))})
    counterfactuals.append(cf)
results["exact_source_replays"]=replays
results["counterfactuals"]=counterfactuals
results["stratified_numeric_trace_review"]=stratified
results["source_code_findings"]={
 "generation_path":"02_generate_causal_eew_dataset.ipynb code cell JSON index2 classes; index3 loop seed42+combined_index; noise uses Phase10 seed/profile.",
 "global_noise_scaling":"raw_std=np.std(filtered_noise) and cur_std=np.std(filtered) across all10000 samples.",
 "global_template_normalization":"env_p/env_s divided by max of their complete future envelope.",
 "actual_window_normalizer":"First100 pre-P/reference samples only. This downstream step is prefix causal but cannot undo generator scaling.",
 "learned_preprocessing":"No learned fitted scaler; per-window context moments. No train/validation pooled fitting.",
 "oracle":"Earthquake windows require exact simulated P; negative windows use a randomly selected reference, not a real P. S is used for eligibility metadata only.",
 "pilot_not_generator":"Notebook01 uses a separate class and cannot by itself certify notebook02 outputs."}
print("Replayed records:",len(replays),"; exact:",sum(r["raw_error"]==0 and r["norm_error"]==0 for r in replays))
print("Counterfactual earthquake seeds:",len(counterfactuals),
      "; max future-noise normalized3s difference:",max(r["future_noise"]["saved3s_normalized_difference"] for r in counterfactuals),
      "; max future-P-coda normalized3s difference:",max(r["future_P_coda"]["saved3s_normalized_difference"] for r in counterfactuals))


Replayed records: 21 ; exact: 21
Counterfactual earthquake seeds: 15 ; max future-noise normalized3s difference: 361.7903289794922 ; max future-P-coda normalized3s difference: 99.57305908203125


In [4]:
# Final classification uses measured failures, not upstream report PASS flags.
def hash_values(obj,location=""):
    pairs=[]
    if isinstance(obj,dict):
        for k,v in obj.items():pairs+=hash_values(v,location+"/"+k)
    elif isinstance(obj,list):
        for i,v in enumerate(obj):pairs+=hash_values(v,location+"/"+str(i))
    elif isinstance(obj,str) and len(obj)==64:
        try:int(obj,16);pairs.append((location,obj.lower()))
        except ValueError:pass
    return pairs
older=hash_values(old_audit)
prior_corroboration={}
for split in ["train","validation","test"]:
    rel=f"data wave/synthetic_waveform_simulator/output/phase11_preprocessed_dataset/{split}.h5"
    prior_corroboration[rel]=[location for location,digest in older if digest==before[rel]]
after={rel:sha(ROOT/rel) for rel in before}
results["protected_after"]=after
results["protected_unchanged"]={rel:after[rel]==value for rel,value in before.items()}
results["older_Phase11_hash_corroboration"]=prior_corroboration
results["v2_registry_matches"]={name:before["eew_causal_v2/dataset/"+name]==item["sha256"].lower()
                               for name,item in readj(V2/"configs/causal_dataset_hashes.json").items()}
numeric_pass=all(all(r["checks"].values()) for r in results["records"].values()) and all(npz_equal.values())
split_pass=not duplicates and not any(split_checks.values()) and results["splits"]["all_pairs_one_eq_one_noise_same_split"]
c_fail=any(r["future_noise"]["saved3s_normalized_difference"]>1e-5 or r["future_P_coda"]["saved3s_normalized_difference"]>1e-5 for r in counterfactuals)
mask_pass=all(r["checks"][f"mask_{t}s"] for r in results["records"].values() for t in [1,2,3])
unchanged=all(results["protected_unchanged"].values())
verdicts={
 "A":{"status":"UNKNOWN" if numeric_pass else "FAIL","detail":"Saved numeric structure/labels PASS if checks true; regression exclusion is UNKNOWN: no explicit regression mask/consumer supplied. Noise -1 must be masked using label==1."},
 "B":{"status":"PASS" if split_pass else "FAIL","detail":"All saved event/pair/trace/background generation keys and exact window hashes checked. Full parents are not saved; whole-parent duplicate claims are limited to deterministic identity mapping, not exhaustive parent waveform comparisons."},
 "C":{"status":"FAIL" if c_fail else "UNKNOWN","detail":"Independent source-matched future-noise/coda perturbations test earlier raw and normalized windows. See counterfactuals."},
 "D":{"status":"PASS" if mask_pass and all(r["checks"]["P_before_S"] and r["checks"]["S_quantization"] and r["checks"]["travel_time_units"] for r in results["records"].values()) else "FAIL",
      "detail":"Simulated onset/interval units and eligibility masks checked; not detected arrivals, calibrated real-world travel times or unique magnitude inversion."},
 "E":{"status":"FAIL" if c_fail else "UNKNOWN","classification":"C. Not causally valid" if c_fail else "A. Causal synthetic oracle-aligned benchmark",
      "detail":"Oracle P-aligned validation/test inputs; future S used as offline eligibility mask only. Positive P always at window index100 after alignment; no streaming trigger exists. Full future-noise scaling defeats causal benchmark classification."},
 "F":{"status":"PASS" if unchanged and all(historical.values()) and all(prior_corroboration.values()) else "FAIL" if not unchanged or not all(historical.values()) else "UNKNOWN",
      "detail":"Before/after audit hashes plus supplied17-file baseline; older waveform audit corroborates original Phase11 files. Baseline is a saved record, not externally authenticated proof covering all repository files."}}
results["audits"]=verdicts
results["decision"]="BLOCKED" if c_fail or not numeric_pass or not split_pass or not unchanged else "CONDITIONAL"
results["actual_records_inspected"]=len(records)
results["samples_in_hdf5_numeric_scans"]="All numeric arrays in all three HDF5 files; all30 NPZ members compared to corresponding HDF5 arrays."
# Reports disagree; classify exactly against independently recomputed eligibility.
eligibility={}
for t in [1,2,3]:
    count=sum(r["eligibility"][str(t)]["eligible_eq"] for r in results["records"].values())
    total=sum(r["eq"] for r in results["records"].values())
    eligibility[str(t)]={"eligible":count,"total_eq":total,"fraction":count/total,
                        "min_SP":min(r["eligibility"][str(t)]["continuous_SP_min"] for r in results["records"].values())}
results["aggregate_eligibility"]=eligibility
maxima={"Sx4_preS_raw":max(r["Sx4_preS_raw"] for r in counterfactuals),
        "Sx4_preS_norm":max(r["Sx4_preS_normalized"] for r in counterfactuals),
        "future_noise_raw3s":max(r["future_noise"]["saved3s_raw_difference"] for r in counterfactuals),
        "future_noise_norm3s":max(r["future_noise"]["saved3s_normalized_difference"] for r in counterfactuals),
        "future_P_coda_raw3s":max(r["future_P_coda"]["saved3s_raw_difference"] for r in counterfactuals),
        "future_P_coda_norm3s":max(r["future_P_coda"]["saved3s_normalized_difference"] for r in counterfactuals),
        "future_S_coda_earlier_raw":max(r["future_S_coda"]["earlier_raw_difference"] for r in counterfactuals),
        "future_S_coda_earlier_norm":max(r["future_S_coda"]["earlier_normalized_difference"] for r in counterfactuals),
        "postprocess_future_perturbation":max(r["post_generation_future_change_prefix_difference"] for r in counterfactuals)}
results["counterfactual_maxima"]=maxima
results["regression_exclusion"]="UNKNOWN: arrays contain noise magnitude -1; no regression loss executed or supplied. Correct future mask must be (labels==1), optionally AND chosen pre-S mask."
results["full_parent_saved_data_verification"]="UNKNOWN for unsaved samples; exact replays verify saved windows for stratified pilots only."
(OUT/"audit_results.json").write_text(json.dumps(normal(results),indent=2,allow_nan=False),encoding="utf-8")
def table(headers,rows):
    return "\n".join(["| "+" | ".join(headers)+" |","|"+"|".join(["---"]*len(headers))+"|"]+
                     ["| "+" | ".join(str(v) for v in row)+" |" for row in rows])
summary=table(["Audit","Status","Observed evidence"],[
 ("A",verdicts["A"]["status"],f"Saved numeric/label checks {'PASS' if numeric_pass else 'FAIL'}; regression masking not implemented/verified."),
 ("B",verdicts["B"]["status"],f"Cross-split membership intersections {sum(split_checks.values())}; exact window duplicate hits {len(duplicates)}."),
 ("C",verdicts["C"]["status"],"Future-noise and future-template normalization change already-observed prefixes."),
 ("D",verdicts["D"]["status"],"P/S metadata, physical interval units and strict eligibility independently checked."),
 ("E",verdicts["E"]["status"],"Oracle-aligned inputs plus upstream future dependence; classification C."),
 ("F",verdicts["F"]["status"],f"{sum(historical.values())}/17 original baseline matches; {sum(results['protected_unchanged'].values())}/{len(before)} protected files unchanged during audit.")])
report=f"""# Final independent causal EEW v2 dataset audit

**Decision: {results["decision"]}. Classification C — not causally valid.** No model training, full dataset regeneration, repairs or protected-file changes. This audit does not approve operational EEW.

## 1. Actual inspection

All **{len(records):,} records**: train11324, validation2426, test2428. Every numeric HDF5 array was checked for finiteness; all labels, IDs, arrival annotations, eligibility masks and nested horizons were compared. All30 NPZ members were compared exactly against HDF5 counterparts. Lengths200/300/400 contain100 pre-reference samples plus100/200/300 post-reference samples, at100Hz; axis order is(samples,time), one waveform channel. Models would need a separately added channel axis.

Stratified replay: **{len(replays)} saved records**, including{len(counterfactuals)} earthquake parameter/seed cases spanning magnitude, depth and short S-P intervals, plus noise. Exact raw/normalized replay matches: **{sum(r["raw_error"]==0 and r["norm_error"]==0 for r in replays)}/{len(replays)}**. Saved-window reproduction links the counterfactual code path to actual data; it does not verify unsaved100-second parents exhaustively.

## 2. Audits A–F

{summary}

Detailed per-split checks, every metadata comparison and measured exceptions are in audit_results.json. Sentinel-1 noise magnitudes and P/S labels are stored correctly where checks pass, but no explicit regression mask or training consumer exists. Noise must be excluded from magnitude regression with labels==1; true pre-S masks on noise mean classification eligibility, not physical arrivals.

## 3. Independent causal counterfactuals

Only class definitions from notebook02 JSON code-cell index2 were executed in memory. The bulk generation loop was not executed. The different pilot class from notebook01 was not treated as proof for notebook02.

{table(["Perturbation","Maximum earlier/raw or saved3s difference","Maximum normalized difference"],[
 ("Future S amplitude x4, before S",f'{maxima["Sx4_preS_raw"]:.9g}',f'{maxima["Sx4_preS_norm"]:.9g}'),
 ("Future noise innovations x4 after P+8s, saved3s",f'{maxima["future_noise_raw3s"]:.9g}',f'{maxima["future_noise_norm3s"]:.9g}'),
 ("P coda envelope x4 only after P+4s, saved3s",f'{maxima["future_P_coda_raw3s"]:.9g}',f'{maxima["future_P_coda_norm3s"]:.9g}'),
 ("S coda envelope x4 only after S+5s, earlier samples",f'{maxima["future_S_coda_earlier_raw"]:.9g}',f'{maxima["future_S_coda_earlier_norm"]:.9g}')])}

**Critical:** notebook02 computes raw_std=np.std(filtered_noise) over the full100-second earthquake background, and cur_std=np.std(filtered) for noise parents. Modifying only future innovations preserves earlier innovations and causal-filter output, but changes global scaling and earlier observed windows. Pre-P normalization can cancel common noise scaling approximately for pure noise; it does **not** cancel the changed signal/noise ratio in earthquake windows.

The P/S envelopes are also divided by their full-record maxima. Perturbing only the future coda envelope can therefore change earlier signal amplitude. These tests change future-tail values before the generator's existing normalization; the earlier unnormalized envelope and ambient realization are held fixed. They do not conflate changing a whole rupture-duration parameter with a future-only intervention.

The narrower S-amplitude-only test passes, so the original future-S-to-noise coupling was removed. It is insufficient to certify the complete generator. Downstream pre-context-only normalization itself passes a direct future-waveform perturbation test (maximum prefix change {maxima["postprocess_future_perturbation"]:.9g}); upstream defects remain.

## 4. Physics, eligibility and discrepancies

{table(["Post-P horizon","Eligible earthquake records","Fraction","Minimum physical S-P seconds"],[(t+"s",f'{r["eligible"]}/{r["total_eq"]}',f'{100*r["fraction"]:.4f}%',f'{r["min_SP"]:.6f}') for t,r in eligibility.items()])}

P indices vary across parent records, but oracle cropping places P at window index100 for every earthquake example. Noise has no P/S label and uses a random reference segment. Validation/test therefore require oracle P timing; they are not a trigger-based streaming benchmark. S timing is needed to filter/evaluate pre-S eligibility, not to calculate the recorded waveform channels.

The earlier verification report's claims of minimum S-P1.06s and eligibility99.6%/98.8% must be compared to the independently measured table above; the completion report instead claims minimum3.89s and100%. This audit's stored values are authoritative for the inspected files. The completion report does not actually contain the explicit oracle acknowledgment attributed to it by the earlier verification.

Magnitude/depth/distance join by source ID; station/source coordinates are available in the linked Phase2 metadata, not input channels. Arrival indices describe simulated source-aligned times, not detected arrivals. At fixed geometry/noise, changing magnitude changes normalized P windows in the numerical probes; this supports simulator sensitivity only, not unique magnitude identifiability across distance/source uncertainty or real calibrated ground-motion units.

## 5. Leakage/split severity and data-interface limits

Observed cross-split source/pair/trace/background-key intersections and exact duplicates are listed above. Noise parent mapping uses Phase10 seed/profile and combined index; earthquake ambient seeds use42+combined_index. No learned global scaler is fitted on train/validation/test. Full parent waveforms are absent, so exhaustive whole-parent content uniqueness remains UNKNOWN even when mapped identities and all saved windows are disjoint.

NPZ arrays contain waveforms and labels separately, not extra label channels, but omit IDs, P/S arrays and1s/2s masks. Their verified row identity requires retaining HDF5/manifest association; noise sentinel-1 is not a ready-to-use regression target. Exact labels and arrival values are not directly concatenated into waveform arrays. The critical issue is upstream future-dependent synthesis, not an observed event-split overlap.

## 6. Evidence and protection

- Actual arrays: ../dataset/eew_dataset_{{train,validation,test}}.h5 and ../dataset/eew_windows_multitask.npz.
- Exact generator and seed/export path: ../02_generate_causal_eew_dataset.ipynb, JSON cells2–4; pilot differences inspected in notebook01 and verification scope in notebook03.
- Source/label association: ../configs/causal_eew_dataset_manifest.csv; ../../data wave/synthetic_waveform_simulator/output/phase2_travel_time/metadata_with_theoretical_arrivals.csv; Phase10 earthquake_noise_mapping.csv; Phase11 split_indices.csv.
- Parameters and registered hashes: ../configs/causal_simulation_parameters.json and causal_dataset_hashes.json.
- Preservation: ../original_files_manifest_initial.json plus older ../../phase1_waveform_cnn_results/reports/waveform_audit.json. The older audit corroborates{sum(bool(x) for x in prior_corroboration.values())}/3 Phase11 hashes. The17-file claim is checked against that supplied historical baseline, not an assertion about every repository file or an externally authenticated history.
- This audit: [dataset_audit.ipynb](dataset_audit.ipynb), [audit_results.json](audit_results.json), containing all before/after hashes, source-matched replay errors and per-seed counterfactuals.

All{len(before)} inspected protected files are compared before/after; unchanged={unchanged}. Existing status, memory, reports, datasets and checkpoints were not edited. **Stop after audit; no repairs or training.**
"""
(OUT/"final_audit_report.md").write_text(report,encoding="utf-8")
print("Final",results["decision"],"; audits",{k:v["status"] for k,v in verdicts.items()})
print("Counterfactual maxima:",maxima)
print("Eligibility:",eligibility,"; protected unchanged:",unchanged)


Final BLOCKED ; audits {'A': 'UNKNOWN', 'B': 'PASS', 'C': 'FAIL', 'D': 'PASS', 'E': 'FAIL', 'F': 'PASS'}
Counterfactual maxima: {'Sx4_preS_raw': 0.0, 'Sx4_preS_norm': 0.0, 'future_noise_raw3s': 0.15472953394055367, 'future_noise_norm3s': 361.7903289794922, 'future_P_coda_raw3s': 5.181565761566162, 'future_P_coda_norm3s': 99.57305908203125, 'future_S_coda_earlier_raw': 2.524754285812378, 'future_S_coda_earlier_norm': 50.555524826049805, 'postprocess_future_perturbation': 0.0}
Eligibility: {'1': {'eligible': 8089, 'total_eq': 8089, 'fraction': 1.0, 'min_SP': 1.0631958838936242}, '2': {'eligible': 8056, 'total_eq': 8089, 'fraction': 0.9959203857089876, 'min_SP': 1.0631958838936242}, '3': {'eligible': 7996, 'total_eq': 8089, 'fraction': 0.9885029051798739, 'min_SP': 1.0631958838936242}} ; protected unchanged: True


In [5]:
# Report precision correction: reconcile continuous physical time with rounded saved S indices.
audit=readj(OUT/"audit_results.json")
rounding=[]
for split in ["train","validation","test"]:
    with h5py.File(DATA/f"eew_dataset_{split}.h5","r") as f:
        eq=f["labels"][:]==1
        ids=f["source_id"].asstr()[:][eq]
        p=f["p_arrival_sample"][:][eq];s=f["s_arrival_sample"][:][eq]
        dt=meta.loc[ids,"theoretical_sp_time_seconds"].to_numpy()
        for tau in [1,2,3]:
            mask=f[f"pre_s_valid_{tau}s"][:][eq]
            for j in np.flatnonzero(mask!=(s-p>tau*100)):
                rounding.append({"split":split,"source_id":ids[j],"horizon_seconds":tau,
                    "physical_SP_seconds":float(dt[j]),"stored_S_minus_P_samples":int(s[j]-p[j]),
                    "stored_mask":bool(mask[j]),"integer_index_strict_test":bool(s[j]-p[j]>tau*100),
                    "interpretation":"Rounding puts S at nominal cutoff; continuous S is later. Saved window ends one sample before cutoff; not actual S contamination."})
audit["arrival_rounding_discrepancies"]=rounding
audit["audits"]["D"]["detail"]+=" Two validation boundary cases disagree with strict tests on rounded integer annotations; continuous source times support the saved masks. Exact cases recorded."
audit["report_discrepancies"]=[
 "Completion report: all horizons100% pre-S and minS-P3.89s contradicted by actual data; measured min1.0631958839s, eligible8089/8056/7996 of8089.",
 "Earlier verification's1.06s and99.6%/98.8% are consistent after rounding; both reports' complete causality PASS is contradicted.",
 "Earlier verification attributes oracle acknowledgment to completion report, but that acknowledgment is absent there.",
 "Two rounded S-index boundary cases require continuous source timestamps to reproduce the strict eligibility predicate."
]
(OUT/"audit_results.json").write_text(json.dumps(normal(audit),indent=2,allow_nan=False),encoding="utf-8")
path=OUT/"final_audit_report.md"
text=path.read_text(encoding="utf-8")
replacement="The completion report's minimum S-P3.89s and100% eligibility at every horizon are contradicted: the actual minimum is1.063195884s; eligible counts are8089/8056/7996 of8089 at1/2/3s. The earlier verification's1.06s and99.6%/98.8% figures are consistent after rounding. Both reports' complete causality PASS is contradicted by this audit. The completion report does not contain the explicit oracle acknowledgment attributed to it by the earlier verification."
text="\n".join(replacement if line.startswith("The earlier verification report's claims") else line for line in text.splitlines())+"\n"
details=" ".join(f'{r["source_id"]} at{r["horizon_seconds"]}s: physical S-P={r["physical_SP_seconds"]:.9f}s, stored difference={r["stored_S_minus_P_samples"]} samples.' for r in rounding)
text=text.replace("## 5. Leakage/split severity and data-interface limits",
    "**Annotation precision:** "+details+" These two validation masks are correct using continuous source times but differ from strict comparisons on rounded integer S indices. The saved windows end before the nominal cutoff, so this is not observed S contamination; consumers must retain the continuous-time definition.\n\n## 5. Leakage/split severity and data-interface limits")
path.write_text(text,encoding="utf-8")
print("Recorded",len(rounding),"annotation-rounding cases; final decision remains",audit["decision"])


Recorded 2 annotation-rounding cases; final decision remains BLOCKED
